# Bottlenecks in grievance disposal, July 2024 - June 2025

Between July 2024 and June 2025, Odisha's Janasunani portal received roughly a little under seven lakh complaints and disposed around 70 per cent of these. For these disposed complaints, the average time to close was 63 days. The goal of this note is to decompose the total time taken into the stops at various offices a complaint traverses during its journey from receipt to closure, and identify the bottlenecks that cause most of the hold-up.

Complaints reach the portal either online or offline, and the records of offline complaints understate how long they spend before reaching an officer. We therefore restrict our analysis to the roughly 3.4 lakh disposed complaints that were filed online. These take 65 days on average, and we find that most of this time is spent in the field. Stage 1, which runs from the receipt of a complaint to the selection of its route, takes 14 days (21 per cent). Stage 2, the action taken in the field for disposal, takes 44 days (68 per cent), and Stage 3, the review by the assigning authority, takes the remaining 7 days. The automations proposed in Janasunani 2.0, which read, deduplicate, remove spam from, classify and summarize complaints, all act on Stage 1. Even if they removed Stage 1 entirely, they would cut at most about a fifth of the time to disposal. This is a real efficiency gain, but it leaves most of the delay untouched.

To see where in the field the time goes, we follow two types of complaints, PMAY housing and old age pensions, along their most common routes. In both, the Block Development Officer's office holds the complaint for most of Stage 2, while the other offices on the route hold it for a few days at most. Better route selection could also help. PMAY complaints that come through the CM Cell spend 38 days in Stage 1 before reaching the field, against 7 days for those sent straight from the Collector to the BDO. But the BDO sits on every common route we observe and holds the complaint for most of the field time, so even the best choice of route leaves the largest delay where it is. The BDO's office is therefore a natural candidate for the bottleneck. However, BDO offices that receive more complaints are not systematically slower, and offices that receive roughly the same number of complaints take very different amounts of time. We do not know what explains these differences, and the portal's records cannot tell us. Understanding them requires a closer look at the resources and constraints of the BDO offices themselves.

In [ ]:
# The two complaint types section 3.2 follows. Edit to change them.
TYPES = {
    "pmay": ("PMAY", "dept_id = 21 AND subcategory = 'IAY/MKY/BPGY/PMAY'"),
    "oap": ("old age pension", "dept_id = 40 AND subcategory = 'OAP/ODP/MBPY/WP'"),
}

In [ ]:
import html

import polars as pl
from IPython.display import Markdown, display

from janasunani.analytics.bottlenecks import check_note, open_lake
from janasunani.analytics.exhibits import (
    draw_office_volume, draw_outcomes_pie, draw_stage_bar, draw_stage_stack, entry_route_groups,
    entry_routes, field_frame, office_volume, outcomes, stage_means, stage_office_splits,
    stage_rows,
)
from janasunani.analytics.figures import Report
from janasunani.analytics.journey import STAGES_THREE
from janasunani.analytics.sample import Sample

con = open_lake()
con.execute("SET enable_progress_bar = false")
pl.Config.set_fmt_str_lengths(120)
# Silent on pass: the CA&GR Analytics Note's published 2024-25 figures.
check_note(con)

# Tables are for a reader, not a debugger: no shape line, no dtype row, no
# quotes around text, blanks for nulls, thousands separators on counts. This
# also applies to tables the exhibits display, and survives export to Word.
def _cell(v):
    if v is None:
        return ""
    return f"{v:,}" if isinstance(v, int) else html.escape(str(v))


def _table_html(df):
    head = "".join(f"<th>{html.escape(c)}</th>" for c in df.columns)
    body = "".join("<tr>" + "".join(f"<td>{_cell(v)}</td>" for v in row) + "</tr>"
                   for row in df.rows())
    return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"


def _table_md(df):
    rows = [list(df.columns), ["---"] * df.width,
            *[[_cell(v) for v in row] for row in df.rows()]]
    return "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)


_fmt = get_ipython().display_formatter.formatters
_fmt["text/html"].for_type(pl.DataFrame, _table_html)
# The Word version is built from a Markdown export, which reads this one.
_fmt["text/markdown"].for_type(pl.DataFrame, _table_md)


STEPS = [
    ("F0", "Complaints filed in 2024-25", "TRUE"),
    ("F1", "Disposed", "is_disposed"),
    ("F2", "Disposed, with a usable closing date",
     "is_disposed AND days_to_close IS NOT NULL"),
]
FINAL = "Final sample: every office on the route identified, stage dates consistent"
FIGDIR = "../outputs/figures"
BDO = "Block Development Officer"


def sample(name, where):
    s = Sample(con, ns=f"{name}_", base=f"year = '2024-25' AND {where}", steps=STEPS)
    s.install_route("F2", "F2")
    s.install_phases("F2", "F2")
    s.STEPS["P4"] = (FINAL, None)
    return s


def by_channel(s, rep):
    groups = [(c, f"ticket_no IN (SELECT ticket_no FROM {s.t('g')} WHERE channel = '{c}')")
              for c in ("Online", "Offline")]
    draw_stage_stack(rep, stage_rows(s, groups, STAGES_THREE, floor=1), "Channel",
                     "Time to disposal by stage, online and offline complaints",
                     "Averages, in days. Bracketed: complaints in the sample.",
                     "01_channel", STAGES_THREE, count_label="Complaints", show_table=False)


def decomposition(s, rep, label):
    means, total = stage_means(s, STAGES_THREE)
    draw_stage_bar(rep, means, total, f"Time to disposal by stage, {label}",
                   f"{s.FUNNEL['P4']:,} disposed complaints. Averages; the three "
                   f"stages add up to {total:.0f} days.", "02_stages", STAGES_THREE)


def routes(s, rep, label):
    groups = [(f"All {label} complaints", "TRUE")] + entry_route_groups(entry_routes(s, top=3))
    rows = stage_rows(s, groups, STAGES_THREE, floor=500)
    kept = [g for g in groups if g[0] in dict(rows)]
    draw_stage_stack(rep, rows, "Route", f"Time to disposal by stage and office, {label}",
                     "Averages, in days. Dashed lines split Stage 2 by the office holding the "
                     "complaint, named under each brace. Bracketed: complaints.", "03_routes", STAGES_THREE,
                     count_label="Complaints", show_table=False,
                     splits=stage_office_splits(s, kept, "Stage 2"))


def volume(everyone, s, rep, label, floor):
    offices = office_volume(everyone, field_frame(s), BDO, floor)
    draw_office_volume(rep, offices, f"Complaints received and Stage 2 days by BDO office, {label}",
                       f"Every BDO office with at least {floor} disposed {label} complaints. "
                       "Received counts every complaint, of any type, that reached the office.",
                       "08_volume", days_label=f"Stage 2 days, {label} complaints")

## 1. Data and sample

As noted earlier, a little under seven lakh complaints were registered in Janasunani between July 2024 and June 2025. Of these, around 69 per cent had been disposed of by the end of the period, while 16 per cent were discarded and a further 16 per cent were still open. Since we can only measure the time taken to dispose of a complaint once it has actually been disposed of, we focus on the roughly 4.7 lakh disposed complaints. Of these, around 4.4 lakh had consistent dates and the time taken at each office identified.

In [ ]:
everyone = sample("all", "TRUE")
assert everyone.FUNNEL["F0"] == 688_301, everyone.FUNNEL["F0"]   # the published total
rep = Report(con, [("Statewide", "TRUE")], FIGDIR, prefix="disp_state_")
draw_outcomes_pie(rep, outcomes(everyone, "F0"), "Complaints filed in 2024-25, by outcome",
                  f"{everyone.FUNNEL['F0']:,} complaints.", "01_outcomes")

Complaints reach the portal either online, through the website, WhatsApp, the mobile app, email or social media, or offline, in person, by letter or at a grievance hearing. Around 3.4 lakh of the complaints in our sample were filed online and around 1 lakh offline. The figure below breaks up the time to disposal for each. Offline complaints appear to spend only 5 days in Stage 1, against 14 days for online complaints. This is unlikely to be because offline complaints are processed faster. An offline complaint is entered into the portal only after it has been received and read at an office, so the time it spends before it is entered is never recorded. Stage 1 times for offline complaints are therefore biased downwards. For the rest of this note, we restrict our analysis to **the roughly 3.4 lakh disposed complaints with consistent dates that were filed online**, which form our final sample.

In [ ]:
by_channel(everyone, rep)

## 2. Three stages

| Stage | | Covers |
|---|---|---|
| **Stage 1** | Processing | From receipt to route selection |
| **Stage 2** | Field action | Action taken for disposal |
| **Stage 3** | Review | Review by the assigning authority |

Stage 3 is zero when a complaint is not sent back for review. The time from the last action to formal closure averages under a day and is left out.

## 3. Time to disposal by stage

The average total time taken to dispose a complaint in our sample can be broken up into the time taken in each of the stages enumerated above. We do this exercise for all complaints filed online, and then for two specific types of complaints: PMAY housing and old age pensions.

### 3.1 All online complaints

The average time for disposing the complaints in our sample of roughly 3.4 lakh online complaints is 65 days. This is then broken up into the time taken for each of the three stages: processing, field action, and review.

In [ ]:
online = sample("online", "channel = 'Online'")
decomposition(online, rep, "online complaints")

Stage 2, the field action, takes 44 of the 65 days (68 per cent). Stage 1, from receipt to route selection, takes 14 days (21 per cent), and Stage 3, review, takes 7 days (11 per cent). The automations proposed in Janasunani 2.0 (reading complaints, deduplicating them, removing spam, classifying them and summarizing them) all act on Stage 1. If implemented perfectly so that the new system removed Stage 1 entirely without affecting quality, we would cut at most about a fifth of the time to disposal: 14 of 65 days. This is a real efficiency gain, but since most of the time taken is spent in the field, where the proposed automation does not reach, we need a better understanding of what the bottlenecks are and whether other forms of AI and automation can help ameliorate them. Choosing a complaint's route better could save further time, both in Stage 1 and by avoiding unnecessary stops in the field. However, as we show below, the BDO's office sits on most routes and accounts for most of the field time whichever route a complaint takes.

### 3.2 By complaint type

To see where in the field the time goes, we follow two types of complaints along their most common routes: PMAY housing complaints, which make up around a quarter of our sample, and old age pension complaints, which make up only around one per cent. For each route we also split Stage 2 by the office holding the complaint. These examples are not meant to be representative of all complaints. Rather, they illustrate concretely where in the field a complaint spends its time.

In [ ]:
subs = {key: sample(key, f"channel = 'Online' AND {where}") for key, (_, where) in TYPES.items()}
label = TYPES["pmay"][0]
display(Markdown(f"#### 3.2.1 {label}"))
routes(subs["pmay"], Report(con, [(label, "TRUE")], FIGDIR, prefix="disp_pmay_"), label)

A PMAY complaint takes 45 days on average, 30 of which are spent in Stage 2. Almost all PMAY complaints follow one of two routes: from the Collector straight to the BDO (79 per cent), or from the CM Cell through the Collector to the BDO and back (21 per cent). On both routes, nearly all of Stage 2 is spent at the BDO's office. The route through the CM Cell takes 62 days, half as long again as the direct route, and most of the difference arises before the complaint reaches the field: 38 days pass in Stage 1, against 7 days on the direct route. This is where better route selection could help: a PMAY complaint that reaches the CM Cell could be sent straight on to the Collector and the BDO, as most PMAY complaints are, avoiding much of this delay. Even so, it would still spend around a month at the BDO's office.

In [ ]:
label = TYPES["oap"][0]
display(Markdown("#### 3.2.2 Old age pensions"))
routes(subs["oap"], Report(con, [(label, "TRUE")], FIGDIR, prefix="disp_oap_"), label)

An old age pension complaint takes 50 days on average, 32 of which are spent in Stage 2. These complaints follow longer routes, passing through the district social security officer before reaching the BDO, and usually returning through it. Even so, the social security officer holds a complaint for only a few days, while the BDO holds it for between 18 and 38 days, depending on the route. Here the choice of route makes little difference to the total, which ranges only from 47 to 51 days, because every route passes through the BDO, who holds the complaint for most of that time.

<div style="page-break-before: always"></div>

## 4. Is the BDO the bottleneck?

In both examples, the BDO's office is where most of the time in the field is spent, and since it sits on nearly every route, choosing better routes cannot by itself remove this delay. A natural explanation is that BDO offices simply receive too many complaints. If this were the case, BDO offices that receive more complaints should take longer to dispose of them. We check this by comparing, for each BDO office, the number of complaints of any type it received in 2024-25 with the average time it took in Stage 2, first for PMAY complaints and then for old age pension complaints.

### 4.1 PMAY

In [ ]:
volume(everyone, subs["pmay"], Report(con, [("PMAY", "TRUE")], FIGDIR, prefix="disp_pmay_"), "PMAY", floor=10)

There is no clear pattern between the number of complaints a BDO office receives and how long it takes on PMAY complaints. At the same time, offices that receive roughly the same number of complaints take very different amounts of time: a fast office takes around 10 days, a slow one more than 50.

### 4.2 Old age pensions

In [ ]:
volume(everyone, subs["oap"], Report(con, [("old age pension", "TRUE")], FIGDIR, prefix="disp_oap_"), "old age pension", floor=10)

Old age pension complaints are far fewer, so each office's average rests on a small number of complaints, and the comparison is noisier. Busier offices, if anything, take slightly longer. But here too, the differences between offices that receive a similar number of complaints are much larger than any difference between busy and quiet offices.

## 5. Conclusion

A disposed online complaint took 65 days on average between July 2024 and June 2025. Most of this time is spent in the field: Stage 2 accounts for 44 days (68 per cent), while Stage 1, from receipt to route selection, accounts for 14 days (21 per cent) and Stage 3, review, for the remaining 7. Within the field, the BDO's office holds the complaint for most of the time. On both of the complaint types we follow, PMAY housing and old age pensions, the other offices on a route hold a complaint for a few days at most, while the BDO holds it for between 15 and 38 days, depending on the route.

Three inferences follow. First, the automations proposed in Janasunani 2.0, which read, deduplicate, remove spam from, classify and summarize complaints, act on Stage 1, and so can cut at most about a fifth of the time to disposal. This is a real efficiency gain, but it leaves most of the delay untouched. Second, better route selection can help where a route adds needless stops, as it does for PMAY complaints that pass through the CM Cell, but the BDO sits on every common route we observe, so no choice of route avoids the largest delay. Third, the delay at the BDO is not simply a matter of volume. BDO offices that receive more complaints are not systematically slower, and offices that receive a similar number of complaints take very different amounts of time. We do not know what explains these differences, and the portal's records cannot tell us. Reducing the time to disposal therefore requires looking beyond the portal, at the resource constraints BDO offices face: how many staff handle grievances, what other duties compete for their time, and what a complaint needs before it can be disposed of.


In [ ]:
con.close()